<div style="text-align:center;"><h1 style="color:yellow;font-family:'Impact';font-size:3em;">Fuentes</h1></div>

## Aparcamientos en Madrid
- Madrid - Aparcamientos ocupaciones (rotación): https://datos.madrid.es/portal/site/egob/menuitem.c05c1f754a33a9fbe4b2e4b284f1a5a0/?action=addValoracion&idValorado=44f9b2213c537610VgnVCM1000008a4a900aRCRD&puntuacion=1&vgnextchannel=374512b9ace9f310VgnVCM100000171f5a0aRCRD&vgnextfmt=default&vgnextoid=44f9b2213c537610VgnVCM1000008a4a900aRCRD&utm_source=chatgpt.com

- Zonas del Servicio de Estacionamiento Regulado SER: https://datos.madrid.es/portal/site/egob/menuitem.c05c1f754a33a9fbe4b2e4b284f1a5a0/?vgnextchannel=374512b9ace9f310VgnVCM100000171f5a0aRCRD&vgnextfmt=default&vgnextoid=b9955cde99be2410VgnVCM1000000b205a0aRCRD&utm_source=chatgpt.com

- Servicio de Estacionamiento Regulado (SER). Tiques de aparcamiento: https://datos.madrid.es/portal/site/egob/menuitem.c05c1f754a33a9fbe4b2e4b284f1a5a0/?vgnextchannel=374512b9ace9f310VgnVCM100000171f5a0aRCRD&vgnextfmt=default&vgnextoid=67663c0a55e16710VgnVCM1000001d4a900aRCRD

    > Estructura del Conjunto de Datos:    
     https://datos.madrid.es/FWProjects/egob/Catalogo/Transporte/Ficheros/Estructura_DS_Tiques_Aparcamiento_SER.pdf

## Información metereológica de AEMET
- Acceso a la API:
    > URL: https://opendata.aemet.es/dist/index.html?
    
    > API Key: ---

- Estaciones AEMET:
    > https://www.aemet.es/es/serviciosclimaticos/datosclimatologicos/valoresclimatologicos#tab1

- Datos históricos:
    > valores-climatologicos (/api/valores/climatologicos/diarios/datos/fechaini/{fechaIniStr}/fechafin/{fechaFinStr}/estacion/{idema}): https://opendata.aemet.es/dist/index.html#/valores-climatologicos/Climatolog%C3%ADas%20diarias.

- Predicciones:
    > predicciones-especificas (/api/prediccion/especifica/municipio/horaria/{municipio}): https://opendata.aemet.es/dist/index.html#/predicciones-especificas/Predicci%C3%B3n%20por%20municipios%20horaria.%20Tiempo%20actual.

## Calendario de grandes eventos
- Agenda turística de Madrid (Madrid Convention Bureau):
    > XML: https://www.esmadrid.com/opendata/agenda_v1_es.xml

    > DCAT: https://datos.madrid.es/egob/catalogo/300028-0-agenda-turismo.dcat

    > Estructura del Dataset: https://datos.madrid.es/FWProjects/egob/Catalogo/Turismo/ficheros/Estructura_DS_agenda_turistica.pdf

- IFEMA:
    > Concentra el 70 % de los congresos de más de 5.000 personas en Madrid; su calendario online cubre la mayor parte de los picos de demanda.
    > 

In [1]:
import pandas as pd
import seaborn as sns

In [2]:
# Dataset de tiques de parquímetro (SER) de Madrid para 2024
df = pd.read_csv('../data/ser_madrid/2024.csv')
df.head()

,matricula_parquimetro,fecha_operacion,fecha_inicio,fecha_fin,cod_distrito,distrito,cod_barrio,barrio,tipo_zona,distintivo,minutos_tique,importe_tique
0,ELPARKING,2024-01-13 12:45:10,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ARGANZUELA,2,ACACIAS,AZUL,ECO,65,"0,30"
1,TELPARK,2024-03-18 14:40:12,2024-03-18 14:40:12,2024-03-18 16:41:12,4,SALAMANCA,2,GOYA,AZUL,C,121,"2,50"
2,EASYPARK,2024-02-27 12:00:12,2024-02-27 12:00:00,2024-02-27 12:20:00,6,TETUAN,4,ALMENARA,AZUL,C,20,"0,60"
3,702430001,2024-03-18 16:45:04,2024-03-18 16:44:00,2024-03-18 17:48:00,2,ARGANZUELA,4,LEGAZPI,VERDE,C,64,"2,00"
4,EASYPARK,2024-02-01 15:29:41,2024-02-01 15:29:00,2024-02-01 15:45:00,9,MONCLOA,3,CIUDAD UNIVERSITARIA,AZUL,ECO,16,"0,05"


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44000 entries, 0 to 43999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   matricula_parquimetro  44000 non-null  object
 1   fecha_operacion        44000 non-null  object
 2   fecha_inicio           44000 non-null  object
 3   fecha_fin              44000 non-null  object
 4   cod_distrito           44000 non-null  int64 
 5   distrito               43649 non-null  object
 6   cod_barrio             44000 non-null  int64 
 7   barrio                 44000 non-null  object
 8   tipo_zona              44000 non-null  object
 9   distintivo             44000 non-null  object
 10  minutos_tique          44000 non-null  int64 
 11  importe_tique          44000 non-null  object
dtypes: int64(3), object(9)
memory usage: 4.0+ MB


## Target:

```%_ocup_barrio(t+1h) = tickets_activos(t+1h) / plazas_totales_barrio```

> Clasificación auxiliar: etiqueta binaria alta_ocup (=1 si % ocup ≥ 85 %)... o un semáforo (3 colores)

### En este caso, el número de tickets activos en la próxima hora cómo se calcularía?

- Un <span style="font-weight:bold;">Ticket Activo</span> es un vehículo que está ocupando una plaza un tiempo determinado.
1. En el dataset tenemos la ````fecha de inicio```` y la ````fecha fin de estacionamiento```` para indicarnos que se trata de un ticket activo.
- Tenemos que identificar cada ticket con un identificador único.
- <mark>Creamos un dataframe con todos los días del año divididos en franjas 15 minutos -además de indicar la hora del día en concreto-</mark>
- (en ese dataframe de las franjas horarias de 15min, añadimos la información del sumatorio de tickets que están activos durante esa hora, junto a otro de información, p.ej. la relativa al barrio)
- (Es decir, no voy a tener en cuenta los tickets que han estado menos de 15 minutos)
---
- Si un ticket se inica en la hora -1, y su periodo de fin no termina hasta más allá de los 15 minutos de la hora 0, ese ticket estará activo durante esa hora. Tampoco importa si la fecha de finalización del ticket se extiende a la hora +1. A efectos de cálculo, durante esa hora, ese ticket cuanta como ticket activo.
y crear un dataframe con todas las franjas de 15min de todo el año.
---
- para calcular el número de tickets activos en una hora determinada, agrupo las franjas de 15min y calculo los identificadores de tickets únicos para esa hora.
(al final tendré en un dataframe una columna hora (con todas las horas del año) y el número de tickets que han estado activos)
---
- cuántos datos debería tener para que el modelo tenga suficientes?
Teniendo en cuenta que el número de plazas disponibles para cada barrio es un dato público y accesible,igual tengo que utilizar todo el dataset finalmente... a traves de Colab no tendré problemas.
Cualquier reducción del dataset original harían desvalancearse los datos puesto que no se podría reducir el número de plazas disponibles en proporción.

In [4]:
# añado un identificador único al dataframe
df['ticket_id'] = range(1,len(df)+1)
df.set_index('ticket_id')
df.head(3)

,matricula_parquimetro,fecha_operacion,fecha_inicio,fecha_fin,cod_distrito,distrito,cod_barrio,barrio,tipo_zona,distintivo,minutos_tique,importe_tique,ticket_id
0,ELPARKING,2024-01-13 12:45:10,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ARGANZUELA,2,ACACIAS,AZUL,ECO,65,"0,30",1
1,TELPARK,2024-03-18 14:40:12,2024-03-18 14:40:12,2024-03-18 16:41:12,4,SALAMANCA,2,GOYA,AZUL,C,121,"2,50",2
2,EASYPARK,2024-02-27 12:00:12,2024-02-27 12:00:00,2024-02-27 12:20:00,6,TETUAN,4,ALMENARA,AZUL,C,20,"0,60",3


In [5]:
# elimino las columnas que no me van a ser útiles.
df.drop(columns=['fecha_operacion', 'cod_distrito', 'distrito', 'importe_tique', 'matricula_parquimetro', 'tipo_zona', 'distintivo'], inplace=True)
df.head(3)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,ticket_id
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,ALMENARA,20,3


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44000 entries, 0 to 43999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   fecha_inicio   44000 non-null  object
 1   fecha_fin      44000 non-null  object
 2   cod_barrio     44000 non-null  int64 
 3   barrio         44000 non-null  object
 4   minutos_tique  44000 non-null  int64 
 5   ticket_id      44000 non-null  int64 
dtypes: int64(3), object(3)
memory usage: 2.0+ MB


In [7]:
# Convierto las fechas/hora a datetime
df['fecha_inicio'] = pd.to_datetime(df['fecha_inicio'])
df['fecha_fin'] = pd.to_datetime(df['fecha_fin'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44000 entries, 0 to 43999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   fecha_inicio   44000 non-null  datetime64[ns]
 1   fecha_fin      44000 non-null  datetime64[ns]
 2   cod_barrio     44000 non-null  int64         
 3   barrio         44000 non-null  object        
 4   minutos_tique  44000 non-null  int64         
 5   ticket_id      44000 non-null  int64         
dtypes: datetime64[ns](2), int64(3), object(1)
memory usage: 2.0+ MB


In [8]:
df.head(20)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,ticket_id
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,ALMENARA,20,3
3,2024-03-18 16:44:00,2024-03-18 17:48:00,4,LEGAZPI,64,4
4,2024-02-01 15:29:00,2024-02-01 15:45:00,3,CIUDAD UNIVERSITARIA,16,5
5,2024-02-01 15:08:36,2024-02-01 19:53:36,3,CIUDAD JARDÍN,285,6
6,2024-03-27 20:00:23,2024-03-30 09:00:23,6,BERRUGUETE,60,7
7,2024-01-23 20:29:00,2024-01-23 20:59:00,2,PROSPERIDAD,30,8
8,2024-01-10 09:00:00,2024-01-10 11:17:00,3,CASTILLEJOS,137,9
9,2024-02-01 15:12:28,2024-02-01 16:37:28,5,LISTA,85,10


In [9]:
# Clasifico los tickets como Activos si superan los 15 minutos
df_activos = df[df['minutos_tique'] >= 15].copy()
df_activos.head(20)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,ticket_id
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,ALMENARA,20,3
3,2024-03-18 16:44:00,2024-03-18 17:48:00,4,LEGAZPI,64,4
4,2024-02-01 15:29:00,2024-02-01 15:45:00,3,CIUDAD UNIVERSITARIA,16,5
5,2024-02-01 15:08:36,2024-02-01 19:53:36,3,CIUDAD JARDÍN,285,6
6,2024-03-27 20:00:23,2024-03-30 09:00:23,6,BERRUGUETE,60,7
7,2024-01-23 20:29:00,2024-01-23 20:59:00,2,PROSPERIDAD,30,8
8,2024-01-10 09:00:00,2024-01-10 11:17:00,3,CASTILLEJOS,137,9
9,2024-02-01 15:12:28,2024-02-01 16:37:28,5,LISTA,85,10


In [13]:
# creo una columna 'slot_inicio' para clasificar cada ticket según la 'fecha_inicio'
df_activos['slot_inicio'] = df_activos['fecha_inicio'].dt.floor('15min')
df_activos.head(3)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,ticket_id,slot_inicio
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1,2024-01-13 12:45:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2,2024-03-18 14:30:00
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,ALMENARA,20,3,2024-02-27 12:00:00


In [15]:
# creo otra columna 'slot_fin' para identificar hasta cuándo está activo cada ticket
df_activos['slot_fin'] = (df_activos['fecha_fin'] - pd.Timedelta(seconds=1)).dt.floor('15min')
df_activos.head()

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,ticket_id,slot_inicio,slot_fin
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1,2024-01-13 12:45:00,2024-01-13 13:45:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2,2024-03-18 14:30:00,2024-03-18 16:30:00
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,ALMENARA,20,3,2024-02-27 12:00:00,2024-02-27 12:15:00
3,2024-03-18 16:44:00,2024-03-18 17:48:00,4,LEGAZPI,64,4,2024-03-18 16:30:00,2024-03-18 17:45:00
4,2024-02-01 15:29:00,2024-02-01 15:45:00,3,CIUDAD UNIVERSITARIA,16,5,2024-02-01 15:15:00,2024-02-01 15:30:00


In [20]:
# creo otra columna 'slots' que contiene todos los slots en los que está cada ticket
def lista_slots(ticket):
    return pd.date_range(
        start = ticket.slot_inicio,
        end = ticket.slot_fin,
        freq = '15min'
    )

df_activos['slots'] = df_activos.apply(lista_slots, axis=1)
df_activos.slots[0]

DatetimeIndex(['2024-01-13 12:45:00', '2024-01-13 13:00:00',
               '2024-01-13 13:15:00', '2024-01-13 13:30:00',
               '2024-01-13 13:45:00'],
              dtype='datetime64[ns]', freq='15min')

In [26]:
# creo un nuevo dataframe que guarde ('explote') una fila por cada slot para cada ticket (para poder agruparlos después)
df_slots = df_activos.explode("slots")
df_slots.head(10)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,ticket_id,slot_inicio,slot_fin,slots
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1,2024-01-13 12:45:00,2024-01-13 13:45:00,2024-01-13 12:45:00
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1,2024-01-13 12:45:00,2024-01-13 13:45:00,2024-01-13 13:00:00
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1,2024-01-13 12:45:00,2024-01-13 13:45:00,2024-01-13 13:15:00
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1,2024-01-13 12:45:00,2024-01-13 13:45:00,2024-01-13 13:30:00
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65,1,2024-01-13 12:45:00,2024-01-13 13:45:00,2024-01-13 13:45:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2,2024-03-18 14:30:00,2024-03-18 16:30:00,2024-03-18 14:30:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2,2024-03-18 14:30:00,2024-03-18 16:30:00,2024-03-18 14:45:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2,2024-03-18 14:30:00,2024-03-18 16:30:00,2024-03-18 15:00:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2,2024-03-18 14:30:00,2024-03-18 16:30:00,2024-03-18 15:15:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121,2,2024-03-18 14:30:00,2024-03-18 16:30:00,2024-03-18 15:30:00


In [43]:
############# DATAFRAME BASE DE OCUPACIÓN #############
# agrupo por slots y barrio
ocupacion = df_slots.groupby(['cod_barrio', 'slots']).size().reset_index(name='tickets_activos')
df_slots[(df_slots['cod_barrio'] == 11)].head(20)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,ticket_id,slot_inicio,slot_fin,slots


,cod_barrio,slots,tickets_activos
0,1,2024-01-02 09:00:00,6
1,1,2024-01-02 09:15:00,6
2,1,2024-01-02 09:30:00,7
3,1,2024-01-02 09:45:00,7
4,1,2024-01-02 10:00:00,7


install tqdm
import tqdm